In [ ]:
# 공통 경로와 패키지
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta
import numpy as np

table_dir = str(PROCESSED_DIR / 'AEGIS-Nursing-Initial-Assessment/de')

# 환자별 노이즈 정보
patient_mapping_df = pd.read_csv(str(PATIENT_MAPPING_PATH), sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
patient_mapping_df['new_pt_id'] = patient_mapping_df['new_pt_id'].astype(str)

patient_id_map = patient_mapping_df.set_index('pt_id')['new_pt_id'].to_dict()
date_offset_map = patient_mapping_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_offset_map = patient_mapping_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

csv_files = glob.glob(os.path.join(table_dir, '**/*.csv'), recursive = True)
print(f"[INFO] 총 처리 대상 파일 수 : {len(csv_files)}")


In [ ]:
# 컬럼 삭제

columns_to_drop = [
    'team', 'bed_num', 'cc_text', 'phx','travel_contry', 'etc_text', 'phx_text', 'surgical_history', 'protector', 'stuff',
    'lab_text', 'transfer_another','trauma', 'ed_ward', 'ad_ward_cd', 'transfer_text', 'dr_cc_cd', 'dr_cc', 'dr_cc_text', 'lmp',
    'edc', 'dced', 'gestation_week', 'gestation_day', 'amniotomy', 'amniotomy_dt', 'show', 'show_dt', 'parity_f', 'parity_p',
    'parity_a1', 'parity_a2', 'parity_l1', 'parity_l2', 'pre_delivery', 'menstrual_regular', 'menstrual_cycle', 'menstrual_from',
    'menstrual_to', 'fht1','fht2','fht3','fht4','cx_cm', 'cx_percent', 'fetal_anomaly', 'fetal_anomaly_result', 'fetal_anomaly_text',
    'mid_sono', 'mid_sono_result', 'mid_sono_text', 'gtt50', 'gtt50_result', 'gtt50_text', 'gtt100', 'gtt100_result', 'gtt100_text',
    'amniocentesis', 'amniocentesis_result', 'amniocentesis_text', 'mh_hiv', 'ex1_hiv', 'ex1_person_id','ex1_person',
    'ex2_hiv', 'ex2_person_id', 'ex2_person', 'complication_text', 'nedis', 'nedis_ed', 'nedis_ad', 'trans_in_cd', 'trans_in',
    'nursing_home_num', 'nursing_home', 'trans_out_cd', 'trans_out', 'dc_trans_out_cd', 'dc_trans_out_nm'
]

for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        total_cols_to_remove = len(columns_to_drop)
        present_columns = [col for col in columns_to_drop if col in nursing_assessment_df.columns]
        removed_cols_count = len(present_columns)

        if removed_cols_count > 0:
            nursing_assessment_df.drop(columns = present_columns, inplace = True)

        nursing_assessment_df.to_csv(file, index=False, encoding = 'utf-8')

    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# opd_nm, depart, ad_ward에서 신부인과, 정신건강의학과 제거

import os
import glob
import pandas as pd

# 키워드 정의
keywords = "산부인과|정신건강의학과|정신의학과"

# 처리 시작
for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        original_len = len(nursing_assessment_df)

        # 각 컬럼에 대해 조건에 맞는 행 제거
        condition = pd.Series([False] * len(nursing_assessment_df))
        for col in ['opd_nm', 'depart', 'ad_ward']:
            if col in nursing_assessment_df.columns:
                condition = condition | nursing_assessment_df[col].fillna('').str.contains(keywords, regex=True)

        # 해당 행 제거
        nursing_assessment_df = nursing_assessment_df[~condition]
        removed = original_len - len(nursing_assessment_df)

        # 덮어쓰기 저장
        nursing_assessment_df.to_csv(file, index=False, encoding='utf-8')

    except Exception as e:
        print(f"[ERROR] 오류 발생 - {file}: {e}")


In [ ]:
# pt_id 치환
for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')

        if 'pt_id' in nursing_assessment_df.columns:
            nursing_assessment_df['pt_id'] = nursing_assessment_df['pt_id'].map(patient_id_map).fillna(nursing_assessment_df['pt_id'])

        nursing_assessment_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] pt_id 오류: {file} → {e}")


In [ ]:
# hos_dt ±7일 ±30분
for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'hos_dt' in nursing_assessment_df.columns:
            nursing_assessment_df['hos_dt'] = pd.to_datetime(nursing_assessment_df['hos_dt'], errors='coerce')
            nursing_assessment_df['hos_dt'] = nursing_assessment_df['hos_dt'] + nursing_assessment_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            nursing_assessment_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# pre_tx_time ±7일 ±30분
for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'pre_tx_time' in nursing_assessment_df.columns:
            nursing_assessment_df['pre_tx_time'] = pd.to_datetime(nursing_assessment_df['pre_tx_time'], errors='coerce')
            nursing_assessment_df['pre_tx_time'] = nursing_assessment_df['pre_tx_time'] + nursing_assessment_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            nursing_assessment_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] pre_tx_time 오류: {file} → {e}")


In [ ]:
# os_dt ±7일 ±30분
for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'os_dt' in nursing_assessment_df.columns:
            nursing_assessment_df['os_dt'] = pd.to_datetime(nursing_assessment_df['os_dt'], errors='coerce')
            nursing_assessment_df['os_dt'] = nursing_assessment_df['os_dt'] + nursing_assessment_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            nursing_assessment_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] os_dt 오류: {file} → {e}")


In [ ]:
# f_ktas_dt ±7일 ±30분
for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'f_ktas_dt' in nursing_assessment_df.columns:
            nursing_assessment_df['f_ktas_dt'] = pd.to_datetime(nursing_assessment_df['f_ktas_dt'], errors='coerce')
            nursing_assessment_df['f_ktas_dt'] = nursing_assessment_df['f_ktas_dt'] + nursing_assessment_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            nursing_assessment_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] f_ktas_dt 오류: {file} → {e}")


In [ ]:
# l_ktas_dt ±7일 ±30분
for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'l_ktas_dt' in nursing_assessment_df.columns:
            nursing_assessment_df['l_ktas_dt'] = pd.to_datetime(nursing_assessment_df['l_ktas_dt'], errors='coerce')
            nursing_assessment_df['l_ktas_dt'] = nursing_assessment_df['l_ktas_dt'] + nursing_assessment_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            nursing_assessment_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] l_ktas_dt 오류: {file} → {e}")


In [ ]:
# ad_date 처리

for file in csv_files:

    try:

        nursing_assessment_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        if 'ad_date' in nursing_assessment_df.columns:

            nursing_assessment_df['ad_date'] = pd.to_datetime(nursing_assessment_df['ad_date'], errors = 'coerce')

            nursing_assessment_df['ad_date'] = nursing_assessment_df['ad_date'] \
                + pd.to_timedelta(nursing_assessment_df['pt_id'].map(date_offset_map).fillna(0).astype(int), unit = "D")

            nursing_assessment_df.to_csv(file, index = False, encoding = 'utf-8')

    except Exception as e:

        print(f"[ERROR] ad_date 오류: {file} -> {e}")


In [ ]:
# dc_dt ±7일 ±30분
for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'dc_dt' in nursing_assessment_df.columns:
            nursing_assessment_df['dc_dt'] = pd.to_datetime(nursing_assessment_df['dc_dt'], errors='coerce')
            nursing_assessment_df['dc_dt'] = nursing_assessment_df['dc_dt'] + nursing_assessment_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            nursing_assessment_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] dc_dt 오류: {file} → {e}")


In [ ]:
# ex1_date 처리

for file in csv_files:

    try:

        nursing_assessment_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        if 'ex1_date' in nursing_assessment_df.columns:

            nursing_assessment_df['ex1_date'] = pd.to_datetime(nursing_assessment_df['ex1_date'], errors = 'coerce')

            nursing_assessment_df['ex1_date'] = nursing_assessment_df['ex1_date'] \
                + pd.to_timedelta(nursing_assessment_df['pt_id'].map(date_offset_map).fillna(0).astype(int), unit = "D")

            print(nursing_assessment_df[~nursing_assessment_df['ex1_date'].isna()]['ex1_date'])

            nursing_assessment_df.to_csv(file, index = False, encoding = 'utf-8')

    except Exception as e:

        print(f"[ERROR] ex1_date 오류: {file} -> {e}")


In [ ]:
# ex2_date 처리

for file in csv_files:

    try:

        nursing_assessment_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        if 'ex2_date' in nursing_assessment_df.columns:

            nursing_assessment_df['ex2_date'] = pd.to_datetime(nursing_assessment_df['ex2_date'], errors = 'coerce')

            nursing_assessment_df['ex2_date'] = nursing_assessment_df['ex2_date'] \
                + pd.to_timedelta(nursing_assessment_df['pt_id'].map(date_offset_map).fillna(0).astype(int), unit = "D")

            print(nursing_assessment_df[~nursing_assessment_df['ex2_date'].isna()]['ex2_date'])

            nursing_assessment_df.to_csv(file, index = False, encoding = 'utf-8')

    except Exception as e:

        print(f"[ERROR] ex2_date 오류: {file} -> {e}")


In [ ]:
# nedis_dt ±7일 ±30분
for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'nedis_dt' in nursing_assessment_df.columns:
            nursing_assessment_df['nedis_dt'] = pd.to_datetime(nursing_assessment_df['nedis_dt'], errors='coerce')
            nursing_assessment_df['nedis_dt'] = nursing_assessment_df['nedis_dt'] + nursing_assessment_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            nursing_assessment_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] nedis_dt 오류: {file} → {e}")


In [ ]:
# ad_certification_date ±7일 ±30분
for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'ad_certification_date' in nursing_assessment_df.columns:
            nursing_assessment_df['ad_certification_date'] = pd.to_datetime(nursing_assessment_df['ad_certification_date'], errors='coerce')
            nursing_assessment_df['ad_certification_date'] = nursing_assessment_df['ad_certification_date'] + nursing_assessment_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            nursing_assessment_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] ad_certification_date 오류: {file} → {e}")


In [ ]:
# 특이 정보 추출 (cc_cd, cc_nm N수 5 이하)

rare_combinations_all = []
for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if {'cc_cd', 'cc_nm'}.issubset(nursing_assessment_df.columns):
            combination_counts_df = nursing_assessment_df.groupby(['cc_cd', 'cc_nm']).size().reset_index(name = 'count')
            rare_combinations = combination_counts_df[combination_counts_df['count'] <= 5].copy()
            rare_combinations['source_file'] = os.path.basename(file)
            rare_combinations_all.append(rare_combinations)
        else:
            print(f"[WARN] 해당 컬럼 없음: {file}")

    except Exception as e:
        print(f"[ERROR] 오류 발생: {file} -> {e}")

if rare_combinations_all:
    rare_combinations_df = pd.concat(rare_combinations_all, ignore_index = True)
    rare_combinations_df = rare_combinations_df.sort_values(by='count', ascending = True).reset_index(drop = True)
else:
    print("[WARN] 조건에 맞는 조합 없음")
print(f"[INFO] 총 count 수: {rare_combinations_df['count'].sum()}")


In [ ]:
# 특이 정보 제거 (cc_cd, cc_nm)

import os
import glob
import pandas as pd

# 희귀 조합 (rare_combinations_df 기준)
rare_combinations = rare_combinations_df[['cc_cd', 'cc_nm']].dropna().drop_duplicates()

for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype={'pt_id': str}, sep=',', quotechar='"', lineterminator='\n')

        if {'cc_cd', 'cc_nm'}.issubset(nursing_assessment_df.columns):
            before_rows = len(nursing_assessment_df)

            joined_table_df = nursing_assessment_df.merge(rare_combinations, on = ['cc_cd', 'cc_nm'], how = 'left', indicator = True)
            filtered_table_df = joined_table_df[joined_table_df['_merge'] == 'left_only'].drop(columns = ['_merge'])

            after_rows = len(filtered_table_df)
            removed_count = before_rows - after_rows

            filtered_table_df.to_csv(file,index = False, encoding = 'utf-8')

            print(f"[INFO] {file}: {removed_count}개 행 제거됨 (남은 행: {after_rows})")

        else:
            print(f"[WARN] 해당 컬럼 없음: {file}")

    except Exception as e:
        print(f"[ERROR] 오류 발생: {file} → {e}")


In [ ]:
# 특이 정보 추출 (dig_nm N수 5 이하)

total_rare_rows = 0
all_rare_value_info = []

for file in csv_files:
    try:
        nursing_assessment_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'dig_nm' in nursing_assessment_df.columns:
            value_counts = nursing_assessment_df['dig_nm'].value_counts(dropna=True)

            rare_values = value_counts[value_counts <= 5]
            rare_value_list = rare_values.index.tolist()
            rare_count = nursing_assessment_df['dig_nm'].isin(rare_value_list).sum()

            nursing_assessment_df.loc[nursing_assessment_df['dig_nm'].isin(rare_value_list), 'dig_nm'] = 'Deidentified'
            nursing_assessment_df.to_csv(file, index = False, encoding = 'utf-8')

            total_rare_rows += rare_count
            rare_info = pd.DataFrame({
                'source_file': os.path.basename(file),
                'dig_nm': rare_values.index,
                'count': rare_values.values
            })
            all_rare_value_info.append(rare_info)

        else:
            print(f"[WARN] 해당 컬럼 없음: {file}")

    except Exception as e:
        print(f"[ERROR] 오류 발생: {file} -> {e}")

if all_rare_value_info:
    if len(all_rare_value_info) == 1:
        rare_values_df = all_rare_value_info[0].copy()
    else:
        rare_values_df = pd.concat(all_rare_value_info, ignore_index = True)

    rare_values_df = rare_values_df.sort_values(by = 'count').reset_index(drop=True)

else:
    print("[WARN] 희귀 dig_nm 값 없음")
